In [35]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 

In [4]:
df=pd.read_csv("breast_cancer.csv")

In [5]:
df.head()

,Sample code number,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1,3,1,1,2
1,1002945,5,4,4,5,7,10,3,2,1,2
2,1015425,3,1,1,1,2,2,3,1,1,2
3,1016277,6,8,8,1,3,4,3,7,1,2
4,1017023,4,1,1,3,2,1,3,1,1,2


In [6]:
df.shape

(683, 11)

In [8]:
df.drop(columns=["Sample code number"] ,inplace=True)

In [10]:
df.head()

,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,5,1,1,1,2,1,3,1,1,2
1,5,4,4,5,7,10,3,2,1,2
2,3,1,1,1,2,2,3,1,1,2
3,6,8,8,1,3,4,3,7,1,2
4,4,1,1,3,2,1,3,1,1,2


In [11]:
df.shape

(683, 10)

In [12]:
X_train, X_test ,y_train ,y_test =train_test_split(df.iloc[ : , :-1] ,df.iloc[ : , -1],test_size=0.2)

In [13]:
X_train.head()

,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses
6,1,1,1,1,2,10,3,1,1
55,8,2,4,1,5,1,5,4,4
497,5,1,1,1,2,1,1,1,1
34,2,1,1,1,2,1,2,1,1
665,5,10,10,10,4,10,5,6,3


In [19]:
scaler= StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [27]:
X_train.shape

(546, 9)

In [71]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train.values)
y_test_tensor = torch.from_numpy(y_test.values)
y_train_tensor = (y_train_tensor == 4).float()
y_test_tensor = (y_test_tensor == 4).float()

In [72]:
X_train_tensor.shape

torch.Size([546, 9])

In [73]:
X_train.dtype

dtype('float64')

In [74]:
y_train_tensor

tensor([0., 1., 0., 0., 1., 1., 1., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1.,
        1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 1., 0., 1., 0., 0., 1., 0., 1.,
        0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 1., 1., 0., 1.,
        0., 1., 1., 0., 1., 0., 1., 1., 0., 0., 1., 1., 0., 0., 0., 0., 1., 0.,
        0., 0., 1., 1., 0., 1., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
        1., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 1., 1., 1., 0., 0., 0., 0.,
        0., 0., 0., 1., 1., 0., 0., 1., 0., 0., 1., 0., 1., 1., 1., 1., 0., 0.,
        1., 0., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 0., 0., 0., 0., 0., 1.,
        1., 0., 0., 1., 1., 1., 0., 0., 1., 1., 0., 0., 0., 1., 1., 0., 0., 1.,
        0., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 1., 0., 1., 0., 1.,
        1., 0., 1., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1.,
        1., 0., 0., 0., 1., 0., 0., 0., 

In [75]:
#defining a model
class NeuralNetwork():
      def __init__(self , X):
          self.weight = torch.rand(X.shape[1],1, dtype=torch.float64 , requires_grad=True)
          self.bias = torch.zeros(1, dtype=torch.float64 , requires_grad =True)

      def forward(self,X):
          Z = torch.matmul(X,self.weight )+self.bias
          y_pred = torch.sigmoid(Z)
          return y_pred

      def loss(self,y_pred,y):
           # Clamp predictions to avoid log(0)
            epsilon = 1e-7
            y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
        
            # Calculate loss
            loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
            return loss

In [76]:
learning_rate=0.1
epochs=25

In [77]:
#training pipeline
#create model
model=NeuralNetwork(X_train_tensor)
#define loop
for epoch in range(epochs):
    #forward pass
    y_pred=model.forward(X_train_tensor)

    #loss
    loss=model.loss(y_pred , y_train_tensor)

    #backward
    loss.backward()

    #parameter update
    with torch.no_grad():
        model.weight -= learning_rate * model.weight.grad
        model.bias -= learning_rate * model.bias.grad

    #zero gradient
    model.weight.grad.zero_()
    model.bias.grad.zero_()

    # print loss in each epoch
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 1.4352321194204647
Epoch: 2, Loss: 1.353141514549736
Epoch: 3, Loss: 1.275398534317442
Epoch: 4, Loss: 1.2023878829631383
Epoch: 5, Loss: 1.1344634719403304
Epoch: 6, Loss: 1.0719276744410955
Epoch: 7, Loss: 1.0150089948039847
Epoch: 8, Loss: 0.9638402143177371
Epoch: 9, Loss: 0.918439965185057
Epoch: 10, Loss: 0.8787012592079569
Epoch: 11, Loss: 0.8443902588164193
Epoch: 12, Loss: 0.8151571583040371
Epoch: 13, Loss: 0.7905585980054337
Epoch: 14, Loss: 0.770088379104087
Epoch: 15, Loss: 0.7532114637168363
Epoch: 16, Loss: 0.7393960103504844
Epoch: 17, Loss: 0.728139410133824
Epoch: 18, Loss: 0.7189862661332924
Epoch: 19, Loss: 0.711538195155904
Epoch: 20, Loss: 0.7054566945508118
Epoch: 21, Loss: 0.7004609525239608
Epoch: 22, Loss: 0.6963225193822759
Epoch: 23, Loss: 0.6928584422405015
Epoch: 24, Loss: 0.689924021382981
Epoch: 25, Loss: 0.6874059199055922


In [78]:
model.weight

tensor([[-0.1212],
        [-0.3336],
        [ 0.4382],
        [-0.3112],
        [ 0.2820],
        [ 0.1793],
        [-0.1349],
        [-0.0358],
        [ 0.2879]], dtype=torch.float64, requires_grad=True)

In [79]:
model.bias

tensor([-0.2497], dtype=torch.float64, requires_grad=True)

In [80]:
#evaluation
with torch.no_grad():
    y_pred=model.forward(X_test_tensor)
    y_pred = (y_pred > 0.5).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f'Accuracy: {accuracy.item()}')
print(y_pred)
print(y_test_tensor)

Accuracy: 0.5792530179023743
tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
     